# Project FORESIGHT — Baseline Forecast

## Objective

Establish a simple baseline for weekly SKU demand forecasting.

The baseline uses the previous week's demand (`lag_1`) as the prediction.

This provides a benchmark against which the machine-learning forecasting model can be evaluated.

### Baseline Strategy

For each SKU:

Predicted Demand = Previous Week Demand

We evaluate the baseline using:

- MAE
- RMSE
- MAPE

Dataset:
`data/processed/weekly_demand.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error


BASE_DIR = Path.cwd().parent

DATA_PATH = BASE_DIR / "data" / "processed" / "weekly_demand.csv"

print("Project:", BASE_DIR)
print("Dataset:", DATA_PATH)

Project: c:\Projects\foresight
Dataset: c:\Projects\foresight\data\processed\weekly_demand.csv


In [2]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Shape: (19191, 20)

Columns:
['sku_id', 'date', 'units_sold', 'revenue', 'avg_price', 'promo_days', 'holiday_days', 'avg_on_hand_units', 'avg_on_order_units', 'category', 'subcategory', 'unit_cost', 'list_price', 'week_start', 'year', 'week_number', 'month', 'quarter', 'promo_ratio', 'margin']


,sku_id,date,units_sold,revenue,avg_price,promo_days,holiday_days,avg_on_hand_units,avg_on_order_units,category,subcategory,unit_cost,list_price,week_start,year,week_number,month,quarter,promo_ratio,margin
0,SKU0001,2022-01-02,19.0,64830.10,3418.625000,0,0,NaN,NaN,Furnishings,Sofas,2110.16,3425.34,2021-12-27,2022,52,1,1,0.000000,24860.835000
1,SKU0001,2022-01-09,28.0,92873.29,3330.847143,0,0,233.0,0.0,Furnishings,Sofas,2110.16,3425.34,2022-01-03,2022,1,1,1,0.000000,34179.240000
2,SKU0001,2022-01-16,30.0,102238.68,3371.474286,0,0,197.0,0.0,Furnishings,Sofas,2110.16,3425.34,2022-01-10,2022,2,1,1,0.000000,37839.428571
3,SKU0001,2022-01-23,39.0,130405.07,3360.420000,0,0,161.0,0.0,Furnishings,Sofas,2110.16,3425.34,2022-01-17,2022,3,1,1,0.000000,48760.140000
4,SKU0001,2022-01-30,39.0,130870.28,3340.902857,3,1,128.0,0.0,Furnishings,Sofas,2110.16,3425.34,2022-01-24,2022,4,1,1,0.428571,47998.971429


In [3]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["sku_id", "date"]).reset_index(drop=True)

print(df.dtypes)

sku_id                           str
date                  datetime64[us]
units_sold                   float64
revenue                      float64
avg_price                    float64
promo_days                     int64
holiday_days                   int64
avg_on_hand_units            float64
avg_on_order_units           float64
category                         str
subcategory                      str
unit_cost                    float64
list_price                   float64
week_start                       str
year                           int64
week_number                    int64
month                          int64
quarter                        int64
promo_ratio                  float64
margin                       float64
dtype: object


In [4]:
df["baseline_prediction"] = (
    df.groupby("sku_id")["units_sold"]
      .shift(1)
)

baseline_df = df.dropna(
    subset=["units_sold", "baseline_prediction"]
).copy()

print("Baseline rows:", len(baseline_df))

display(
    baseline_df[
        [
            "date",
            "sku_id",
            "units_sold",
            "baseline_prediction"
        ]
    ].head(10)
)

Baseline rows: 18991


,date,sku_id,units_sold,baseline_prediction
1,2022-01-09,SKU0001,28.0,19.0
2,2022-01-16,SKU0001,30.0,28.0
3,2022-01-23,SKU0001,39.0,30.0
4,2022-01-30,SKU0001,39.0,39.0
5,2022-02-06,SKU0001,31.0,39.0
6,2022-02-13,SKU0001,46.0,31.0
7,2022-02-20,SKU0001,33.0,46.0
8,2022-02-27,SKU0001,27.0,33.0
9,2022-03-06,SKU0001,23.0,27.0
10,2022-03-13,SKU0001,24.0,23.0


In [5]:
y_true = baseline_df["units_sold"]
y_pred = baseline_df["baseline_prediction"]

mae = mean_absolute_error(y_true, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

non_zero = y_true != 0

mape = (
    np.mean(
        np.abs(
            (
                y_true[non_zero]
                - y_pred[non_zero]
            )
            / y_true[non_zero]
        )
    )
    * 100
)

print("BASELINE PERFORMANCE")
print("=" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")

BASELINE PERFORMANCE
MAE  : 6.5078
RMSE : 12.8912
MAPE : 43.16%


In [6]:
sku_baseline = []

for sku, group in baseline_df.groupby("sku_id"):

    y_true_sku = group["units_sold"]
    y_pred_sku = group["baseline_prediction"]

    mae_sku = mean_absolute_error(
        y_true_sku,
        y_pred_sku
    )

    rmse_sku = np.sqrt(
        mean_squared_error(
            y_true_sku,
            y_pred_sku
        )
    )

    sku_baseline.append(
        {
            "sku_id": sku,
            "MAE": mae_sku,
            "RMSE": rmse_sku
        }
    )

sku_baseline = pd.DataFrame(sku_baseline)

display(
    sku_baseline.sort_values("MAE").head(10)
)

,sku_id,MAE,RMSE
74,SKU0075,0.788462,1.083087
193,SKU0194,1.000000,1.253566
13,SKU0014,1.057692,1.351637
196,SKU0197,1.142857,1.690309
110,SKU0111,1.250000,1.605280
19,SKU0020,1.250000,1.669869
37,SKU0038,1.269231,1.669869
79,SKU0080,1.317308,1.816061
130,SKU0131,1.346154,1.715316
44,SKU0045,1.461538,1.975815


In [7]:
OUTPUT_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "baseline_results.csv"
)

sku_baseline.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)

Saved: c:\Projects\foresight\data\processed\baseline_results.csv


## Conclusion

The previous-week-demand model provides a simple benchmark.

The machine-learning model in `03_model.ipynb` should outperform this baseline in terms of MAE and RMSE.

The baseline is intentionally simple and does not use:

- Long-term demand history
- Rolling demand statistics
- Seasonality
- Promotions
- Holidays
- Product pricing